# LegaLoom-Env: GRPO Training (Quick Run ~45 min on free T4)

Produces **real** reward curves from actual training. No synthetic data.

**Steps:**
1. Install + clone
2. Measure untrained baseline
3. Train 15 steps on task_hard (no hints, procedural invoices)
4. Plot real curves from trainer.state.log_history
5. Measure trained scores
6. Save artifacts for README

In [ ]:
# Cell 1: Install (3-5 min)
!pip install unsloth trl>=0.12.0 openenv-core==0.2.3 fastapi uvicorn pydantic httpx openai pyyaml datasets matplotlib --quiet
print('✓ Installed')

In [ ]:
# Cell 2: Clone from HF Space
import subprocess, sys, os
!git clone https://huggingface.co/spaces/aarav0202/legaloom-env /content/legaloom_env 2>&1 | tail -3
sys.path.insert(0, '/content/legaloom_env')
os.chdir('/content/legaloom_env')
print('✓ Cloned')

In [ ]:
# Cell 3: Load model (5-8 min)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
print(f'✓ Model: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M trainable params')

In [ ]:
# Cell 4: Measure REAL baseline — untrained model, same prompt as training
# This is the 'before' in before/after. Uses rollout_episode, not random numbers.
from unsloth import FastLanguageModel as FLM
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

# Switch to inference mode for baseline measurement
FLM.for_inference(model)

print('Measuring untrained baseline (5 episodes per task)...')
baseline_scores = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 47):  # 5 episodes
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.3)
        scores.append(result['final_reward'])
        print(f'  {task_id} seed={seed}: {result["final_reward"]:.3f}')
    baseline_scores[task_id] = sum(scores)/len(scores)
    print(f'  → {task_id} avg: {baseline_scores[task_id]:.3f}')

print(f'\nBaseline average: {sum(baseline_scores.values())/4:.3f}')
print('✓ Real baseline measured')

In [ ]:
# Cell 5: Switch back to training mode and run GRPO (15-20 min)
# Training on task_hard: no hints, procedural invoices, full episode rollouts
from unsloth import FastLanguageModel as FLM
from train_grpo import build_training_dataset, episode_reward_fn
from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported

# Switch back to training mode
FLM.for_training(model)

dataset = build_training_dataset(
    task_ids=['task_hard'],
    examples_per_task=60,  # enough for 15 steps
)
print(f'Dataset: {len(dataset)} examples')

def make_reward_fn(tid):
    def fn(prompts, completions, **kwargs):
        return episode_reward_fn(prompts, completions, task_id=tid, **kwargs)
    return fn

config = GRPOConfig(
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=512,  # longer for multi-step sequences
    beta=0.01,
    logging_steps=1,
    max_steps=15,
    output_dir='./grpo_output',
    report_to='none',
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    save_strategy='no',
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    reward_funcs=[make_reward_fn('task_hard')],
    processing_class=tokenizer,
)

print('Starting GRPO training (15 steps, task_hard, no hints)...')
trainer.train()
log_history = trainer.state.log_history
print(f'\n✓ Training complete. {len(log_history)} log entries.')

In [ ]:
# Cell 6: Plot REAL reward curve from trainer.state.log_history
import matplotlib.pyplot as plt
import json

rewards = [e['reward'] for e in log_history if 'reward' in e]
losses  = [e['loss']   for e in log_history if 'loss'   in e]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Reward curve
ax = axes[0]
steps = list(range(1, len(rewards)+1))
ax.plot(steps, rewards, 'b-o', markersize=5, linewidth=1.5, alpha=0.7, label='Episode reward')
if len(rewards) >= 3:
    ma = [sum(rewards[max(0,i-2):i+1])/min(i+1,3) for i in range(len(rewards))]
    ax.plot(steps, ma, 'r-', linewidth=2.5, label='3-step moving avg')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold (0.5)')
ax.set_xlabel('Training Step', fontsize=11)
ax.set_ylabel('Reward', fontsize=11)
ax.set_title('GRPO Training — task_hard (no hints)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

# Loss curve
ax2 = axes[1]
if losses:
    ax2.plot(range(1, len(losses)+1), losses, 'g-o', markersize=4, linewidth=1.5)
ax2.set_xlabel('Training Step', fontsize=11)
ax2.set_ylabel('Loss', fontsize=11)
ax2.set_title('GRPO Training — Loss', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.suptitle('LegaLoom-Env: Real GRPO Training (Qwen2.5-3B, task_hard, 15 steps)', fontsize=11)
plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the actual numbers — paste these into README
print('\n=== REWARD LOG (paste into README) ===')
for i, r in enumerate(rewards):
    print(f'  step {i+1}: reward={r:.4f}')

with open('training_log.json', 'w') as f:
    json.dump(log_history, f, indent=2, default=str)
print('\n✓ Saved reward_curves.png + training_log.json')

In [ ]:
# Cell 7: Measure REAL trained scores (5 episodes per task)
FLM.for_inference(model)

print('Measuring trained model scores (5 episodes per task)...')
trained_scores = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 47):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    trained_scores[task_id] = sum(scores)/len(scores)
    print(f'  {task_id}: before={baseline_scores[task_id]:.3f}  after={trained_scores[task_id]:.3f}  Δ={trained_scores[task_id]-baseline_scores[task_id]:+.3f}')

print(f'\nBaseline avg: {sum(baseline_scores.values())/4:.3f}')
print(f'Trained avg:  {sum(trained_scores.values())/4:.3f}')
print(f'Lift:         {(sum(trained_scores.values())-sum(baseline_scores.values()))/4:+.3f}')
print('\n✓ Real before/after measured')

In [ ]:
# Cell 8: Before/After bar chart
import matplotlib.pyplot as plt
import json

tasks  = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
labels = ['Easy', 'Medium', 'Hard', 'Expert']
before = [baseline_scores[t] for t in tasks]
after  = [trained_scores[t]  for t in tasks]

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(tasks))
b1 = ax.bar([i-0.2 for i in x], before, 0.35, label='Before (Qwen2.5-3B, untrained)', color='#e74c3c', alpha=0.85)
b2 = ax.bar([i+0.2 for i in x], after,  0.35, label='After (GRPO, 15 steps, task_hard)', color='#2ecc71', alpha=0.85)

for bar in b1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in b2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Average Score (5 episodes)', fontsize=11)
ax.set_title('LegaLoom-Env: Before vs After GRPO Training\n(Qwen2.5-3B-Instruct, task_hard, 15 steps, no hints)', fontsize=12, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='Success threshold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
plt.show()

# Save training_scores.json
with open('training_scores.json', 'w') as f:
    json.dump({'baseline': baseline_scores, 'trained': trained_scores,
               'model': 'Qwen2.5-3B-Instruct', 'steps': 15, 'task_trained_on': 'task_hard'}, f, indent=2)

print('✓ Saved before_after.png + training_scores.json')

In [ ]:
# Cell 9: Download all artifacts + print README update
from google.colab import files
import json

files.download('reward_curves.png')
files.download('before_after.png')
files.download('training_scores.json')
files.download('training_log.json')

with open('training_scores.json') as f:
    scores = json.load(f)

print('\n=== PASTE THIS INTO README.md Results table ===')
print('| Task | Baseline | After GRPO | Δ |')
print('|------|----------|-----------|---|')
for t in ['task_easy','task_medium','task_hard','task_expert']:
    b = scores['baseline'][t]
    a = scores['trained'][t]
    print(f'| {t} | {b:.2f} | {a:.2f} | {a-b:+.2f} |')
b_avg = sum(scores['baseline'].values())/4
a_avg = sum(scores['trained'].values())/4
print(f'| **Average** | **{b_avg:.2f}** | **{a_avg:.2f}** | **{a_avg-b_avg:+.2f}** |')
print()
print('Now: commit reward_curves.png, before_after.png, training_scores.json to the repo.')
print('Then update README.md with the table above.')